In [32]:
import yfinance as yf
import pandas as pd
from pathlib import Path

In [33]:
START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

TICKERS = {
    "GOLD": "GC=F",
    "DXY": "DX-Y.NYB",
    "US10Y": "^TNX",
    "VIX": "^VIX",
    "SP500": "^GSPC",
    "OIL": "CL=F",
}

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DIR / "gold_macro_raw.csv"

In [34]:
def download_close_series(ticker: str, column_name: str, start: str, end: str) -> pd.DataFrame:
    df = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False
    )

    if df.empty:
        raise ValueError(f"Data kosong untuk ticker {ticker}")

    df = df[["Close"]].copy()
    df.columns = [column_name]
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    return df

In [35]:
data = {}

for name, ticker in TICKERS.items():
    print(f"Downloading {name} ({ticker})...")
    df = download_close_series(ticker, name, START_DATE, END_DATE)
    data[name] = df
    print(f"{name}: {df.shape[0]} rows")

GOLD: 4022 rows
DXY: 4024 rows
US10Y: 4021 rows
VIX: 4023 rows
SP500: 4023 rows
OIL: 4023 rows


In [36]:
# gunakan GOLD sebagai anchor date
df_all = data["GOLD"].copy()

for name, df in data.items():
    if name == "GOLD":
        continue
    df_all = df_all.join(df, how="left")

df_all = df_all.sort_index()
df_all.index.name = "DATE"

df_all.head()

,GOLD,DXY,US10Y,VIX,SP500,OIL
DATE,,,,,,
2010-01-04,1117.699951,77.529999,3.841,20.040001,1132.989990,81.510002
2010-01-05,1118.099976,77.620003,3.755,19.350000,1136.520020,81.769997
2010-01-06,1135.900024,77.489998,3.808,19.160000,1137.140015,83.180000
2010-01-07,1133.099976,77.910004,3.822,19.059999,1141.689941,82.660004
2010-01-08,1138.199951,77.470001,3.808,18.129999,1144.979980,82.750000


In [37]:
print("Missing values sebelum handling:")
print(df_all.isna().sum())

Missing values sebelum handling:
GOLD     0
DXY      1
US10Y    3
VIX      2
SP500    2
OIL      0
dtype: int64


In [38]:
# forward fill variabel makro agar selaras dengan tanggal trading emas
macro_cols = [col for col in df_all.columns if col != "GOLD"]
df_all[macro_cols] = df_all[macro_cols].ffill()

# buang baris awal jika masih ada missing setelah ffill
df_all = df_all.dropna().copy()

print("Missing values sesudah handling:")
print(df_all.isna().sum())

Missing values sesudah handling:
GOLD     0
DXY      0
US10Y    0
VIX      0
SP500    0
OIL      0
dtype: int64


In [39]:
print("Date range final:")
print(df_all.index.min(), "to", df_all.index.max())

print("\nShape final:")
print(df_all.shape)

print("\nPreview:")
display(df_all.head())
display(df_all.tail())

Date range final:
2010-01-04 00:00:00 to 2025-12-30 00:00:00

Shape final:
(4022, 6)

Preview:


,GOLD,DXY,US10Y,VIX,SP500,OIL
DATE,,,,,,
2010-01-04,1117.699951,77.529999,3.841,20.040001,1132.989990,81.510002
2010-01-05,1118.099976,77.620003,3.755,19.350000,1136.520020,81.769997
2010-01-06,1135.900024,77.489998,3.808,19.160000,1137.140015,83.180000
2010-01-07,1133.099976,77.910004,3.822,19.059999,1141.689941,82.660004
2010-01-08,1138.199951,77.470001,3.808,18.129999,1144.979980,82.750000


,GOLD,DXY,US10Y,VIX,SP500,OIL
DATE,,,,,,
2025-12-23,4482.799805,97.940002,4.169,14.00,6909.790039,58.380001
2025-12-24,4480.600098,97.980003,4.136,13.47,6932.049805,58.349998
2025-12-26,4529.100098,98.019997,4.136,13.60,6929.939941,56.740002
2025-12-29,4325.100098,98.040001,4.116,14.20,6905.740234,58.080002
2025-12-30,4370.100098,98.239998,4.130,14.33,6896.240234,57.950001


In [40]:
df_all.to_csv(OUTPUT_PATH)

print(f"File saved to: {OUTPUT_PATH}")

File saved to: ..\data\raw\gold_macro_raw.csv
